In [12]:
%load_ext autoreload
%autoreload 2
    
import os
import sys
import pandas as pd
import xarray as xr
import numpy as np
from dask.distributed import Client
import cartopy.crs as ccrs
from cartopy.util import add_cyclic_point
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize
import warnings
warnings.filterwarnings('ignore')
from collections import defaultdict
from pathlib import Path
sys.path.append('/home/563/sc1326/repos/cdrmip_extremes')
from cdrmip_extremes.configs import data_dir, proc_dir
from cdrmip_extremes import processing, utils, ext_freq, load_processed,load_data
from cdrmip_extremes.scenariomip import utils_ssp

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
client = Client(memory_limit=None,threads_per_worker=1,n_workers=28)

In [3]:
exceedances_ssp = load_processed.load_ext_data('ScenarioMIP',varnames=['cold_exceedances','heat_exceedances'])

Loading: /home/563/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/extremes/cold_exceedances/ACCESS-CM2/ssp126/ACCESS-CM2_ssp126_cold_exceedances.nc
Loading: /home/563/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/extremes/cold_exceedances/ACCESS-ESM1-5/ssp126/ACCESS-ESM1-5_ssp126_cold_exceedances.nc
Loading: /home/563/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/extremes/cold_exceedances/ACCESS-ESM1-5/ssp534-over/ACCESS-ESM1-5_ssp534-over_cold_exceedances.nc
Loading: /home/563/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/extremes/cold_exceedances/CESM2-WACCM/ssp126/CESM2-WACCM_ssp126_cold_exceedances.nc
Loading: /home/563/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/extremes/cold_exceedances/CESM2-WACCM/ssp534-over/CESM2-WACCM_ssp534-over_cold_exceedances.nc
Loading: /home/563/sc1326/repos/cdrmip_extremes/data/processed/ScenarioMIP/extremes/cold_exceedances/CNRM-ESM2-1/ssp534-over/CNRM-ESM2-1_ssp534-over_cold_exceedances.nc
Loading: /

In [6]:
exceedances_cdr = load_data.load_ext_freq_data()

In [31]:
exceedances_cdr

{'Multi-Model Median': {'cold_exceedances': <xarray.Dataset> Size: 2MB
  Dimensions:  (branch: 2, gwl: 3, lat: 90, lon: 180)
  Coordinates:
    * branch   (branch) <U9 72B 'ramp_up' 'ramp_down'
    * gwl      (gwl) float64 24B 1.5 2.0 3.0
    * lat      (lat) float64 720B -89.0 -87.0 -85.0 -83.0 ... 83.0 85.0 87.0 89.0
    * lon      (lon) float64 1kB 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0
      extrema  <U4 16B ...
  Data variables:
      sigma1   (branch, gwl, lat, lon) float64 778kB ...
      sigma2   (branch, gwl, lat, lon) float64 778kB ...
      sigma3   (branch, gwl, lat, lon) float64 778kB ...,
  'heat_exceedances': <xarray.Dataset> Size: 2MB
  Dimensions:  (branch: 2, gwl: 3, lat: 90, lon: 180)
  Coordinates:
    * branch   (branch) <U9 72B 'ramp_up' 'ramp_down'
    * gwl      (gwl) float64 24B 1.5 2.0 3.0
    * lat      (lat) float64 720B -89.0 -87.0 -85.0 -83.0 ... 83.0 85.0 87.0 89.0
    * lon      (lon) float64 1kB 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0
 

In [14]:
differences_ssp, agreement_ssp = utils_ssp.calc_gwl_differences(exceedances_ssp)

In [41]:
differences_cdr, agreement_cdr = defaultdict(dict), {}

for ext_type in ['heat', 'cold']:
    frequencies = {model: exc_dict[f'{ext_type}_exceedances']
                   for model, exc_dict in exceedances_cdr.items()}
    
    differences, agreement = ext_freq.calc_gwl_differences(frequencies)
    agreement_cdr[ext_type] = agreement
    
    for model, ds in differences.items():
        differences_cdr[model][f'{ext_type}_exceedances'] = ds

differences_cdr = dict(differences_cdr)

## Plot

In [59]:
ds_ssp = differences_ssp['ACCESS-ESM1-5']['ssp534-over']['heat_exceedances'].sigma2
ds_cdr = differences_cdr_by_model['ACCESS-ESM1-5']['heat_exceedances'].sel(gwl=1.5).sigma2.drop_vars(
    ['gwl','extrema'],errors='ignore'
)

In [60]:
ds_cdr

<xarray.DataArray 'sigma2' (lat: 90, lon: 180)> Size: 130kB
array([[ -9.52380952,  -9.52380952,  -9.52380952, ...,  -9.52380952,
         -9.52380952,  -9.52380952],
       [  0.        ,   0.        ,   0.        , ...,   0.        ,
          0.        ,   0.        ],
       [  0.        ,   0.        ,  -4.76190476, ...,   4.76190476,
          4.76190476,   4.76190476],
       ...,
       [  0.        ,   0.        ,   0.        , ...,   0.        ,
          0.        ,  -4.76190476],
       [-14.28571429,  -9.52380952,  -9.52380952, ...,  -9.52380952,
         -9.52380952, -14.28571429],
       [ -4.76190476,  -4.76190476,  -4.76190476, ...,  -4.76190476,
         -4.76190476,  -4.76190476]])
Coordinates:
  * lat      (lat) float64 720B -89.0 -87.0 -85.0 -83.0 ... 83.0 85.0 87.0 89.0
  * lon      (lon) float64 1kB 0.0 2.0 4.0 6.0 8.0 ... 352.0 354.0 356.0 358.0

In [65]:
ds_ssp

<xarray.DataArray 'sigma2' (lat: 90, lon: 180)> Size: 130kB
array([[-5.48387097e+00, -5.72580645e+00, -5.72580645e+00, ...,
        -5.64516129e+00, -5.56451613e+00, -5.48387097e+00],
       [-2.01612903e+00, -1.77419355e+00, -1.77419355e+00, ...,
        -2.09677419e+00, -2.17741935e+00, -2.25806452e+00],
       [-2.74193548e+00, -2.50000000e+00, -3.62903226e+00, ...,
        -3.30645161e+00, -3.38709677e+00, -2.66129032e+00],
       ...,
       [ 2.41935484e-01,  8.06451613e-02,  0.00000000e+00, ...,
        -1.11022302e-16,  8.06451613e-02,  1.61290323e-01],
       [-4.03225806e-01, -4.03225806e-01, -4.83870968e-01, ...,
        -4.83870968e-01, -4.03225806e-01, -4.03225806e-01],
       [-8.06451613e-01, -8.06451613e-01, -8.06451613e-01, ...,
        -5.64516129e-01, -7.25806452e-01, -8.06451613e-01]])
Coordinates:
  * lat      (lat) float64 720B -89.5 -87.49 -85.48 -83.47 ... 85.48 87.49 89.5
  * lon      (lon) float64 1kB -179.5 -177.5 -175.5 -173.5 ... 175.5 177.5 179.5
    extrema  <U4 16B 'heat'

In [61]:
corr = xr.corr(ds_ssp,ds_cdr,dim=['lat','lon'])

In [64]:
corr.values

array(nan)